                                                              Assignment-4

Numba Programming

Name: Arman Garg

Roll_no: 102356002

Grp:- P1B

Q1. You are given a large NumPy array of size 5000000 initialized with random values. Compute the following element-wise operate on: f(x)=x2+3x+5,
for each element in the array and convert it into a CUDA kernel using Numba.Compare performance diﬀerence of CPU with GPU.
a. Modify the kernel to use float32 and float64

In [20]:
import numpy as np
import time
from numba import cuda

# Create large array
n = 5_000_000
arr = np.random.rand(n).astype(np.float64)

# CPU version
def cpu_func(x):
    return x**2 + 3*x + 5

start = time.time()
cpu_result = cpu_func(arr)
cpu_time = time.time() - start

print("CPU Time:", cpu_time)

@cuda.jit
def gpu_func(x, out):
    i = cuda.grid(1)
    if i < x.size:
        out[i] = x[i]**2 + 3*x[i] + 5

# Allocate memory
d_x = cuda.to_device(arr)
d_out = cuda.device_array_like(arr)

threads = 256
blocks = (n + threads - 1) // threads

start = time.time()
gpu_func[blocks, threads](d_x, d_out)
cuda.synchronize()
gpu_time = time.time() - start

gpu_result = d_out.copy_to_host()

print("GPU Time:", gpu_time)
print("Speedup:", cpu_time / gpu_time)

arr32 = arr.astype(np.float32)

d_x32 = cuda.to_device(arr32)
d_out32 = cuda.device_array_like(arr32)

start = time.time()
gpu_func[blocks, threads](d_x32, d_out32)
cuda.synchronize()
print("Float32 GPU Time:", time.time() - start)

CPU Time: 0.034630775451660156
GPU Time: 0.06783151626586914
Speedup: 0.5105410782197914
Float32 GPU Time: 0.11125636100769043


Q2. Implement and benchmark a 1-D histogram computa8on for 1 million
random values in Python using Numba. Compare diﬀerent approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness.

In [21]:
import numpy as np
import time
from numba import njit

data = np.random.rand(1_000_000)
bins = 10

def hist_python(data, bins):
    hist = [0]*bins
    for x in data:
        idx = int(x * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1
    return hist

start = time.time()
h1 = hist_python(data, bins)
print("Python Time:", time.time() - start)

start = time.time()
h2, _ = np.histogram(data, bins=bins)
print("NumPy Time:", time.time() - start)

@njit
def hist_numba(data, bins):
    hist = np.zeros(bins)
    for x in data:
        idx = int(x * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1
    return hist

start = time.time()
h3 = hist_numba(data, bins)
print("Numba Time:", time.time() - start)


Python Time: 0.4541923999786377
NumPy Time: 0.019884586334228516
Numba Time: 0.15599370002746582


Q3. Write a function monte_carlo_pi(nsamples) that estimates the value of π by
generating random x, y coordinates between 0 and 1 and checking if they fall inside a unit circle (x2 + y2 < 1).

a. Implement the function in pure Python first and later create a Numba
version.

b. Program a script to compare the execution time for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).

c. Why does the very first execution of the Numba func8on take slightly
longer than the second execution?

In [22]:
import random
import time

def monte_carlo_pi_python(n):
    inside = 0
    for _ in range(n):
        x, y = random.random(), random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / n

n = 5_000_000

start = time.time()
pi1 = monte_carlo_pi_python(n)
t1 = time.time() - start

print("Python π:", pi1)
print("Time:", t1)

#Numba Version
from numba import njit
import numpy as np

@njit
def monte_carlo_pi_numba(n):
    inside = 0
    for i in range(n):
        x = np.random.random()
        y = np.random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / n

start = time.time()
pi2 = monte_carlo_pi_numba(n)
t2 = time.time() - start

print("Numba π:", pi2)
print("Time:", t2)
print("Speedup:", t1 / t2)

Python π: 3.1411648
Time: 1.7525157928466797
Numba π: 3.1419224
Time: 0.1698288917541504
Speedup: 10.319303004012275


Q4. You have a 1D NumPy array representing pixel intensities (values 0–255). You need to increase the brightness of every pixel by 20%, but ensure no value exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize
decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the time diﬀerence when the work is automa8cally split
across your CPU cores.

d. What happens if you try to pass a list instead of a NumPy array to this
function?

In [23]:
import numpy as np,time
from numba import vectorize

@vectorize(['int64(int64)'])
def f(x):
    y=int(x*1.2)
    return 255 if y>255 else y

a=np.random.randint(0,256,10000000)

t=time.time()
r=f(a)
t1=time.time()-t

@vectorize(['int64(int64)'],target='parallel')
def g(x):
    y=int(x*1.2)
    return 255 if y>255 else y

t=time.time()
r2=g(a)
t2=time.time()-t

print("normal time:",round(t1,4))
print("parallel time:",round(t2,4))
print("sample output:",r[:10])

try:
    f([10,20,30])
    print("list works")
except:
    print("list fails / converts internally")

normal time: 0.0251
parallel time: 0.0228
sample output: [228  92 144 122 164 128 162  70  19 192]
list works


Q5. Write Python code to generate synthe8c training data of 100,000 samples, 10 features and binary labels {-1, +1}. Implement binary logis8c regression using the mathematical formula for gradient descent:

a. Using standard NumPy (without Numba)

b. Using Numba JIT accelera8on

c. Compare correctness and performance.

In [24]:
import numpy as np,time
from numba import njit

X=np.random.randn(100000,10)
y=np.random.choice([-1,1],100000)

def f1(X,y):
    w=np.zeros(10)
    for _ in range(10):
        z=X@w
        g=-(y/(1+np.exp(y*z)))@X/len(y)
        w-=0.01*g
    return w

@njit
def f2(X,y):
    w=np.zeros(10)
    for _ in range(10):
        g=np.zeros(10)
        for i in range(len(y)):
            z=0
            for j in range(10): z+=X[i][j]*w[j]
            v=-y[i]/(1+np.exp(y[i]*z))
            for j in range(10): g[j]+=v*X[i][j]
        g/=len(y)
        w-=0.01*g
    return w

t=time.time()
w1=f1(X,y)
t1=time.time()-t

t=time.time()
w2=f2(X,y)
t2=time.time()-t

print("numpy time:",round(t1,4))
print("numba time:",round(t2,4))
print("speedup:",round(t1/t2,2))
print("correct:",np.allclose(w1,w2))

numpy time: 0.0252
numba time: 0.303
speedup: 0.08
correct: True


Q6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [25]:
import numpy as np
from numba import cuda

@cuda.jit
def add(a,b,c):
    i,j=cuda.grid(2)
    if i<a.shape[0] and j<a.shape[1]:
        c[i][j]=a[i][j]+b[i][j]

n=1024
A=np.random.rand(n,n)
B=np.random.rand(n,n)

dA=cuda.to_device(A)
dB=cuda.to_device(B)
dC=cuda.device_array((n,n))

t=(16,16)
b=(n//16,n//16)

add[b,t](dA,dB,dC)

C=dC.copy_to_host()

print("sample:",C[0][:5])

sample: [1.42203489 0.94964798 1.62741004 1.05379641 1.04960535]
